# Diagnóstico YOLO + ByteTrack para vehículos pequeños desde dron

Compara detección sin tracking, ByteTrack y BoT-SORT. El modelo COCO preentrenado detecta `car`, `motorcycle`, `bus` y `truck`, pero puede fallar con vehículos diminutos vistos desde gran altura. Ajustar el tracker no corrige por sí solo cajas mal localizadas.

## 1. Instalar dependencias

In [ ]:
%pip install -U ultralytics opencv-python pyyaml matplotlib pandas tqdm

## 2. Configuración: cambia la ruta del video

In [ ]:
from pathlib import Path
import cv2, yaml, time
import pandas as pd
from IPython.display import display, Video
from ultralytics import YOLO
from ultralytics.utils import ROOT
from tqdm.auto import tqdm

VIDEO_PATH = Path("input1.mp4")  # CAMBIAR
MODEL_PATH = "yolo11s.pt"  # usa "yolo11n.pt" si priorizas velocidad
OUT_DIR = Path("bytetrack_experiment")
OUT_DIR.mkdir(exist_ok=True)
VEHICLE_CLASSES = [2, 3, 5, 7]  # COCO: car, motorcycle, bus, truck
IMGSZ = 960
DEVICE = None  # None automático; Mac Apple Silicon puede probar "mps"

if not VIDEO_PATH.is_file():
    raise FileNotFoundError(f"Edita VIDEO_PATH; no se encontró: {VIDEO_PATH}")
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened(): raise RuntimeError(f"No se pudo abrir {VIDEO_PATH}")
FPS = cap.get(cv2.CAP_PROP_FPS) or 30.0
WIDTH, HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
NFRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f"{WIDTH}x{HEIGHT}, {FPS:.2f} FPS, {NFRAMES} frames, {NFRAMES/FPS:.1f} s")

## 3. Revisar YOLO sin tracking
Si las cajas ya son incorrectas en esta etapa, ByteTrack no las arreglará.

In [ ]:
from PIL import Image
model = YOLO(MODEL_PATH)
print({i: model.names[i] for i in VEHICLE_CLASSES})
sample_ids = sorted(set([0, NFRAMES//4, NFRAMES//2, 3*NFRAMES//4]))
cap = cv2.VideoCapture(str(VIDEO_PATH))
rows = []
for frame_id in sample_ids:
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
    ok, frame = cap.read()
    if not ok: continue
    r = model.predict(frame, imgsz=IMGSZ, conf=0.10, iou=0.60,
                      classes=VEHICLE_CLASSES, device=DEVICE, verbose=False)[0]
    display(Image.fromarray(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)))
    if r.boxes is not None and len(r.boxes):
        for b, c, k in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int)):
            x1,y1,x2,y2=b
            rows.append({"frame":frame_id,"class":model.names[k],"confidence":float(c),
                         "width_px":x2-x1,"height_px":y2-y1})
cap.release()
df = pd.DataFrame(rows)
display(df.head(30) if not df.empty else "No se detectaron vehículos en las muestras.")
if not df.empty: display(df.groupby("class").agg(n=("class","size"), confidence_media=("confidence","mean"), ancho_mediano=("width_px","median"), alto_mediano=("height_px","median")).round(2))

## 4. Crear YAMLs personalizados
ByteTrack no tiene compensación del movimiento de cámara. BoT-SORT sí, por eso merece una comparación en video de dron.

In [ ]:
tracker_dir = OUT_DIR / "trackers"
tracker_dir.mkdir(exist_ok=True)
cfg_dir = Path(ROOT) / "cfg" / "trackers"

def make_cfg(source, name, overrides):
    cfg = yaml.safe_load((cfg_dir / source).read_text())
    cfg.update(overrides)
    path = tracker_dir / name
    path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print(path, "\n", path.read_text())
    return path

byte_yaml = make_cfg("bytetrack.yaml", "bytetrack_drone.yaml", {
    "track_high_thresh": 0.25, "track_low_thresh": 0.08,
    "new_track_thresh": 0.30, "track_buffer": 30,
    "match_thresh": 0.80, "fuse_score": True
})
bot_yaml = make_cfg("botsort.yaml", "botsort_drone.yaml", {
    "track_high_thresh": 0.25, "track_low_thresh": 0.08,
    "new_track_thresh": 0.30, "track_buffer": 30,
    "match_thresh": 0.80, "fuse_score": True,
    "gmc_method": "sparseOptFlow", "with_reid": False
})

## 5. Ejecutar un tracker y guardar el video\n\nEn `track()`, usamos `conf=0.05` para que las detecciones de baja confianza puedan llegar al tracker; `track_low_thresh=0.08` decide cuáles pueden participar en la segunda asociación. Si el ruido es excesivo, sube `conf` a `0.10` o `0.15` y compara.

In [ ]:
def run_tracking(tracker_yaml, run_name, conf=0.05, imgsz=960):
    out = OUT_DIR / f"{run_name}.mp4"
    m = YOLO(MODEL_PATH)
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    writer = cv2.VideoWriter(str(out), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w,h))
    if not writer.isOpened(): raise RuntimeError(f"No se pudo crear {out}")
    rows=[]; start=time.time(); n=0
    try:
        stream = m.track(source=str(VIDEO_PATH), tracker=str(tracker_yaml),
                         persist=True, stream=True, conf=conf, iou=0.60,
                         imgsz=imgsz, classes=VEHICLE_CLASSES,
                         device=DEVICE, verbose=False)
        for r in tqdm(stream, desc=run_name):
            frame = r.plot()
            if frame.shape[1] != w or frame.shape[0] != h: frame=cv2.resize(frame,(w,h))
            writer.write(frame); n += 1
            boxes = r.boxes
            count = 0 if boxes is None else len(boxes)
            ids = 0 if boxes is None or boxes.id is None else len(set(boxes.id.int().cpu().tolist()))
            rows.append({"frame":n-1,"detections":count,"unique_ids_in_frame":ids})
    finally: writer.release()
    stats=pd.DataFrame(rows); stats.to_csv(OUT_DIR/f"{run_name}_stats.csv",index=False)
    summary={"run":run_name,"conf":conf,"imgsz":imgsz,"frames":n,
             "seconds":round(time.time()-start,2),
             "processing_FPS":round(n/(time.time()-start),2),
             "mean_detections_per_frame":round(stats.detections.mean(),2),
             "video":str(out.resolve())}
    return summary, out, stats

byte_summary, byte_video, byte_stats = run_tracking(byte_yaml, "bytetrack_conf015_img960", conf=0.05, imgsz=960)
display(pd.DataFrame([byte_summary]))

## 6. Comparar BoT-SORT

In [ ]:
bot_summary, bot_video, bot_stats = run_tracking(bot_yaml, "botsort_conf015_img960", conf=0.05, imgsz=960)
display(pd.DataFrame([byte_summary, bot_summary]))

## 7. Prueba opcional para objetos más pequeños
Puede tardar más. Compara las cajas en los mismos fotogramas.

In [ ]:
hi_summary, hi_video, hi_stats = run_tracking(byte_yaml, "bytetrack_conf010_img1280", conf=0.15, imgsz=1280)
display(pd.DataFrame([byte_summary, bot_summary, hi_summary]))

## 8. Visualizar salida

In [ ]:
display(Video(str(byte_video.resolve()), embed=False, width=850))
# Para BoT-SORT: display(Video(str(bot_video.resolve()), embed=False, width=850))

## Interpretación
- Más cajas no significa necesariamente mejor precisión; puede aumentar falsos positivos.
- `conf` más bajo recupera candidatos pero también ruido.
- `imgsz=1280` puede ayudar con objetos pequeños, con mayor coste de memoria y tiempo.
- BoT-SORT compensa movimiento de cámara y puede mejorar la asociación, pero no arregla cajas malas.
- Si los vehículos pequeños siguen sin aparecer, considera *tiling* (inferencia sobre recortes solapados) o entrenar un detector con imágenes aéreas anotadas. Para medir precision/recall de forma objetiva se necesitan anotaciones de referencia.